In [1]:
from enum import Enum


class Library(str, Enum):
    DEEPEVAL = 'deepeval'
    RAGAS = 'ragas'
    TRULENS = 'trulens'
    PHOENIX = 'phoenix'
    MLFLOW = 'mlflow'
    RAGCHECKER = 'ragchecker'
    LLAMA_INDEX = 'llama_index'
    TONIC_VALIDATE = 'tonic_validate'

In [2]:
import logging


logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
from eval_fusion_core.utils.loaders import load_evaluation_inputs
# from eval_fusion_test.settings import get_openai_settings

In [4]:
from eval_fusion_deepeval.evaluator import DeepEvalEvaluator
from eval_fusion_deepeval.metrics import DeepEvalMetric
from eval_fusion_llama_index.evaluator import LlamaIndexEvaluator
from eval_fusion_llama_index.metrics import LlamaIndexMetric
from eval_fusion_mlflow.evaluator import MlFlowEvaluator
from eval_fusion_mlflow.metrics import MlFlowMetric
from eval_fusion_phoenix.evaluator import PhoenixEvaluator
from eval_fusion_phoenix.metrics import PhoenixMetric
from eval_fusion_ragas.evaluator import RagasEvaluator
from eval_fusion_ragas.metrics import RagasMetric
from eval_fusion_ragchecker.evaluator import RagCheckerEvaluator
from eval_fusion_ragchecker.metrics import RagCheckerMetric
from eval_fusion_tonic_validate.evaluator import TonicValidateEvaluator
from eval_fusion_tonic_validate.metrics import TonicValidateMetric
from eval_fusion_trulens.evaluator import TruLensEvaluator
from eval_fusion_trulens.metrics import TruLensMetric

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
INFO:phoenix.config:📋 Ensuring phoenix working directory: /Users/lukapanic/.phoenix
INFO:phoenix.inferences.inferences:Dataset: phoenix_inferences_0e007f78-4646-4932-b65a-e0ad044b8050 initialized
W0730 01:25:21.764000 8334 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/munch/__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
from decouple import config
from eval_fusion_core.models.settings import EvalFusionEMSettings, EvalFusionLLMSettings
from eval_fusion_openai import OpenAIEM, OpenAILLM


def get_openai_settings(llm_name: str, em_name: str):
    vendor = 'lambda' if llm_name.startswith('llama') else 'openai'
    vendor = vendor.upper()

    llm_settings = EvalFusionLLMSettings(
        base_type=OpenAILLM,
        kwargs={
            'model_name': llm_name,
            'base_url': config(f'{vendor}_BASE_URL'),
            'api_key': config(f'{vendor}_API_KEY'),
        },
    )
    em_settings = EvalFusionEMSettings(
        base_type=OpenAIEM,
        kwargs={
            'model_name': em_name,
            'base_url': config('OPENAI_BASE_URL'),
            'api_key': config('OPENAI_API_KEY'),
        },
    )

    return llm_settings, em_settings

In [6]:
import asyncio
import inspect

from functools import partial

from eval_fusion_core.base import EvalFusionBaseEvaluator, EvalFusionBaseMetric
from eval_fusion_core.models import EvaluationOutput, TokenUsage
from pydantic import BaseModel


class EvaluationResult(BaseModel):
    outputs: list[EvaluationOutput]
    token_usage: TokenUsage | tuple[TokenUsage, TokenUsage] | None


async def a_test_evaluator(
    evaluator_cls: type[EvalFusionBaseEvaluator],
    metrics: list[EvalFusionBaseMetric],
    llm_name: str,
    em_name: str,
) -> EvaluationResult:
    llm_settings, em_settings = get_openai_settings(llm_name, em_name)
    inputs = load_evaluation_inputs('../assets/amnesty_qa.json')

    signature = inspect.signature(evaluator_cls)
    has_em_settings = 'em_settings' in signature.parameters
    evaluator = (
        evaluator_cls(llm_settings, em_settings)
        if has_em_settings
        else evaluator_cls(llm_settings)
    )

    with evaluator:
        if evaluator_cls in (MlFlowEvaluator, TruLensEvaluator):
            # loop = asyncio.get_event_loop()
            # func = partial(
            #     evaluator.evaluate,
            #     inputs, metrics=metrics, feature=None, include_reason=False,
            # )
            # outputs = await loop.run_in_executor(None, func)

            outputs = evaluator.evaluate(
                inputs,
                metrics=metrics,
                feature=None,
                include_reason=False,
            )

        else:
            outputs = await evaluator.a_evaluate(
                inputs,
                metrics=metrics,
                feature=None,
                include_reason=False,
            )

    return EvaluationResult(outputs=outputs, token_usage=evaluator.token_usage)

In [7]:
import os


async def a_evaluate_by(
    llm_name: str,
    em_name: str,
    metric: EvalFusionBaseMetric,
    library: str,
    evaluator_cls: EvalFusionBaseEvaluator,
):
    result = await a_test_evaluator(evaluator_cls, [metric], llm_name, em_name)
    dir_path = f'outputs/{llm_name}/{library}'
    os.makedirs(dir_path, exist_ok=True)

    with open(f'{dir_path}/{metric.value}.json', 'w') as file:
        data = result.model_dump_json(indent=4)
        file.write(data)

## LLM

In [8]:
# llm_name = 'gpt-4o-mini'
# llm_name = 'gpt-4o'
llm_name = 'llama3.1-405b-instruct-fp8'
em_name = 'text-embedding-3-large'

### faithfulness

In [9]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

In [10]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=RagasEvaluator,
)

In [9]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3262: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


2025/07/30 01:25:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'custom_llm'.
Created version '1' of model 'custom_llm'.
INFO:backoff:Backing off _check_health(...) for 4.9s (httpx.ConnectError: [Errno 61] Connection refused)
2025/07/30 01:25:42 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

2025/07/30 01:29:03 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
library = 'ragchecker'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagCheckerMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=RagCheckerEvaluator,
)

2025-07-30 01:14:10.899 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for response of 1 RAG results.
2025-07-30 01:14:10.907 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for response of 1 RAG results.
2025-07-30 01:14:10.922 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for response of 1 RAG results.
  0%|          | 0/1 [00:00<?, ?it/s]2025-07-30 01:14:10.931 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for response of 1 RAG results.

  0%|          | 0/1 [00:00<?, ?it/s]2025-07-30 01:14:10.948 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for response of 1 RAG results.2025-07-30 01:14:10.957 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for response of 1 RAG results.


  0%|          | 0/1 [00:00<?, ?it/s]2025-07-30 01:14:10.974 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for response of 1 RAG 

In [13]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.FAITHFULNESS,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

### context_precision/contextual_precision

In [14]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.CONTEXT_PRECISION,
    library=library,
    evaluator_cls=RagasEvaluator,
)

In [15]:
library = 'ragchecker'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagCheckerMetric.CONTEXT_PRECISION,
    library=library,
    evaluator_cls=RagCheckerEvaluator,
)

2025-07-30 01:14:45.519 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for gt_answer of 1 RAG results.
  0%|          | 0/1 [00:00<?, ?it/s]2025-07-30 01:14:45.528 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for gt_answer of 1 RAG results.
2025-07-30 01:14:45.534 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for gt_answer of 1 RAG results.
2025-07-30 01:14:45.540 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for gt_answer of 1 RAG results.

2025-07-30 01:14:45.546 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for gt_answer of 1 RAG results.
  0%|          | 0/1 [00:00<?, ?it/s]2025-07-30 01:14:45.552 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for gt_answer of 1 RAG results.

2025-07-30 01:14:45.558 | INFO     | ragchecker.evaluator:extract_claims:113 - Extracting claims for gt_answer of 1 RAG results.
2025-07-30 01:14:45.5

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.CONTEXTUAL_PRECISION,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

### context_recall/contextual_recall

In [ ]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.CONTEXT_RECALL,
    library=library,
    evaluator_cls=RagasEvaluator,
)

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.CONTEXTUAL_RECALL,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

### answer_relevance/answer_relevancy/response_relevancy

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.ANSWER_RELEVANCE,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.ANSWER_RELEVANCY,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.ANSWER_RELEVANCY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

In [ ]:
library = 'ragas'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=RagasMetric.RESPONSE_RELEVANCY,
    library=library,
    evaluator_cls=RagasEvaluator,
)

### answer_correctness/correctness

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.ANSWER_CORRECTNESS,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3262: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


2025/07/30 00:58:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'custom_llm'.
Created version '1' of model 'custom_llm'.
INFO:backoff:Backing off _check_health(...) for 1.3s (httpx.ConnectError: [Errno 61] Connection refused)
INFO:backoff:Backing off _check_health(...) for 1.1s (httpx.ConnectError: [Errno 61] Connection refused)
2025/07/30 00:58:37 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:373: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

2025/07/30 01:01:27 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.CORRECTNESS,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

### relevance/relevancy

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.RELEVANCE,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3262: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


2025/07/30 01:01:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'custom_llm'.
Created version '1' of model 'custom_llm'.
INFO:backoff:Backing off _check_health(...) for 1.7s (httpx.ConnectError: [Errno 61] Connection refused)
INFO:backoff:Backing off _check_health(...) for 2.8s (httpx.ConnectError: [Errno 61] Connection refused)
2025/07/30 01:01:48 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

2025/07/30 01:01:53 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

2025/07/30 01:03:47 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
library = 'phoenix'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=PhoenixMetric.RELEVANCE,
    library=library,
    evaluator_cls=PhoenixEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.RELEVANCY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

LlamaIndexEvaluationTaskResult(score=1.0, reason='YES.', error=None, time=0.9034743340162095)
LlamaIndexEvaluationTaskResult(score=1.0, reason='YES.', error=None, time=1.8202548749977723)
LlamaIndexEvaluationTaskResult(score=1.0, reason='YES', error=None, time=1.8210252500139177)
LlamaIndexEvaluationTaskResult(score=1.0, reason='YES', error=None, time=0.8711220830155071)
LlamaIndexEvaluationTaskResult(score=0, reason='NO.', error=None, time=1.8209897499764338)
LlamaIndexEvaluationTaskResult(score=0, reason='NO', error=None, time=0.8659224169969093)
LlamaIndexEvaluationTaskResult(score=1.0, reason='YES.', error=None, time=1.8277456249925308)
LlamaIndexEvaluationTaskResult(score=1.0, reason='YES.', error=None, time=1.8279098750208504)
LlamaIndexEvaluationTaskResult(score=0, reason='NO.', error=None, time=0.9278061660006642)
LlamaIndexEvaluationTaskResult(score=0, reason='NO', error=None, time=1.811852250015363)
LlamaIndexEvaluationTaskResult(score=1.0, reason='YES', error=None, time=1.82

### answer_similarity/semantic_similarity

In [ ]:
library = 'mlflow'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=MlFlowMetric.ANSWER_SIMILARITY,
    library=library,
    evaluator_cls=MlFlowEvaluator,
)

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3262: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


2025/07/30 01:04:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'custom_llm'.
Created version '1' of model 'custom_llm'.
INFO:backoff:Backing off _check_health(...) for 0.5s (httpx.ConnectError: [Errno 61] Connection refused)
INFO:backoff:Backing off _check_health(...) for 0.2s (httpx.ConnectError: [Errno 61] Connection refused)
INFO:backoff:Backing off _check_health(...) for 3.4s (httpx.ConnectError: [Errno 61] Connection refused)
2025/07/30 01:04:04 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

2025/07/30 01:04:12 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

2025/07/30 01:06:08 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
library = 'tonic_validate'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=TonicValidateMetric.ANSWER_SIMILARITY,
    library=library,
    evaluator_cls=TonicValidateEvaluator,
)

In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.SEMANTIC_SIMILARITY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

### context_relevance/context_relevancy/contextual_relevancy

In [ ]:
library = 'trulens'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=TruLensMetric.CONTEXT_RELEVANCE,
    library=library,
    evaluator_cls=TruLensEvaluator,
)

/Users/lukapanic/Documents/Projects/eval-fusion/libs/test/.venv/lib/python3.12/site-packages/trulens/core/database/sqlalchemy.py:134: UserWarning: SQLite in-memory may not be threadsafe. See https://www.sqlite.org/threadsafe.html
  warnings.warn(
INFO:alembic.runtime.migration:Context impl SQLiteImpl.
INFO:alembic.runtime.migration:Will assume non-transactional DDL.


🦑 Initialized with db url sqlite:///:memory: .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


✅ In context_relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In context_relevance, input context will be set to __record__.app.retriever.get_context.rets[:] .


In [ ]:
library = 'llama_index'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=LlamaIndexMetric.CONTEXT_RELEVANCY,
    library=library,
    evaluator_cls=LlamaIndexEvaluator,
)

In [ ]:
library = 'deepeval'
await a_evaluate_by(
    llm_name=llm_name,
    em_name=em_name,
    metric=DeepEvalMetric.CONTEXTUAL_RELEVANCY,
    library=library,
    evaluator_cls=DeepEvalEvaluator,
)